# KCVanguar Pre Test - Multi-class tabular classification
## Aplication of CatBoost

This is a storng model when the data consist of numeric dan catagorical colums.

In [ ]:
import pandas as pd
from catboost import CatBoostClassifier

# Load data

# KAGGLE PATHS
# train = pd.read_csv('/kaggle/input/pre-test-kcvanguard2026/train.csv')
# test = pd.read_csv('/kaggle/input/pre-test-kcvanguard2026/test.csv')

# LOCAL PATHS
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

: 

### Data Clean Up

We need a cleaning function to handle missing value and inconsistent writing.
The following code replace:
1. missing value with unknown
2. converts all text to lowercase
3. removes bracket []
4. normalize whitespace

In [ ]:
# Function
def cleanText(text):
    if pd.isna(text):
        return "unknown"
    text = str(text).lower()
    text = text.replace('[', '').replace(']', '')
    text = text.replace('1', '')   
    return " ".join(text.split())

# Implementation
for df in [train, test]:
    df['genre'] = df['genre'].apply(cleanText)
    df['creator'] = df['creator'].apply(cleanText)
    df['titleClean'] = df['title'].apply(cleanText)

### Define Features

`featureCols` what the model is allowed to look at
`catFeatureCols` which of those are categorical

In [ ]:
# Feature set for CatBoost
featureCols = [
    'release_year', 'episode_count', 'num_seasons', 'duration_minutes',
    'episode_format', 'animation_type', 'genre', 'creator', 'country'
]

catFeatureCols = ['episode_format', 'animation_type', 'genre', 'creator', 'country']

xTrain = train[featureCols].fillna(-1)
yTrain = train['network']

### CatBoost Configuration
The CatBoost configuration defines how the model learns multiclass patterns from your mixed numeric and categorical features.

#### Config
`loss_function="MultiClass"` tells the model this is a multiclass classification problem
`eval_metric="Accuracy"` lets you monitor how well it performs during training
`iterations=3000` sets the maximum number of trees
`learning_rate=0.05` controls how aggressively each tree updates the model (smaller values are safer but require more iterations)
`depth=6` limits tree complexity to avoid overfitting
`and l2_leaf_reg=20` adds regularization to make splits more conservative
`random_strength=10` along with `bootstrap_type="Bernoulli"` and  `subsample=0.8`, introduces randomness and row sampling to improve generalization

After configuring the model, `model.fit(...)` learns the mapping from feature patterns to the target network labels, automatically handling categorical columns. Finally, `model.predict(xTest)` applies the trained model to unseen test data to generate the predicted network for submission.

In [ ]:
# Model
model = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="Accuracy",
    iterations=3000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=20,
    random_strength=10,
    bootstrap_type="Bernoulli",
    subsample=0.8,
    max_ctr_complexity=2,
    one_hot_max_size=30,
    od_type="Iter",
    od_wait=120,
    random_seed=42,
    verbose=200
)

model.fit(xTrain, yTrain, cat_features=catFeatureCols)

xTest = test[featureCols].fillna(-1)
predictions = model.predict(xTest)

Make submission file

In [ ]:
# Submission 
sub = pd.DataFrame({'id': test['id'], 'network': predictions.flatten()})
sub.to_csv('submission.csv', index=False)

print("finish - :{D")